# Building Research agent with tools without MCP
Companion notebook for the **Building Research agent with tools without MCP** section of Chapter 5.

This takes the research agent from Chapter 4 and converts its worker nodes (`search_wikipedia`, `search_web`, `fetch_url`) into LangChain **tools**, then lets an LLM decide which ones to call -- the pattern MCP later standardizes across processes.

> Add `OPENAI_API_KEY` (and `TAVILY_API_KEY`) to your `.env` for a real run. Without them the notebook still completes -- the agent node reports the missing key instead of crashing.

In [1]:
import os
from typing import Annotated, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, AIMessage

load_dotenv()


class AgentState(BaseModel):
    """State for the tool-calling agent."""

    messages: Annotated[list[AnyMessage], add_messages] = Field(
        default_factory=list, description="The conversation messages"
    )
    query: str = Field(default="", description="The user's research question")
    final_report: str = Field(default="", description="Final research report")

`messages` uses `add_messages` as its reducer -- it merges (appends to) the list instead of overwriting it, same idea as the reducers from Chapter 4.

Convert the three Chapter 4 worker nodes into tools with `@tool`:

In [2]:
from langchain_core.tools import tool
from langchain_community.retrievers import WikipediaRetriever

wiki = WikipediaRetriever()


@tool
def search_wikipedia(query: str) -> list[dict]:
    """Search Wikipedia for information about a topic.

    Args:
        query: The search query to look up on Wikipedia

    Returns:
        A list of dictionaries containing URL and content from Wikipedia articles
    """
    if not query or query.strip() == "":
        return []

    results = []
    try:
        docs = wiki.invoke(query)
        for doc in docs[:3]:  # limit to 3 documents
            url = doc.metadata.get("url", "")
            if not url or not url.startswith("http"):
                title = doc.metadata.get("title", "").replace(" ", "_")
                url = f"https://en.wikipedia.org/wiki/{title}"
            results.append({"url": url, "content": doc.page_content[:500]})
    except Exception as e:
        print(f"Wikipedia search error: {e}")
        return []

    return results

C:\Users\himan\AppData\Local\Temp\ipykernel_38600\2262230525.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [3]:
from tavily import TavilyClient


@tool
def search_web(query: str) -> list[dict]:
    """Search the web using Tavily for current information.

    Args:
        query: The search query to look up on the web

    Returns:
        A list of dictionaries containing URL and content from web search results
    """
    if not query or query.strip() == "":
        return []

    web_results = []
    try:
        tavily_api_key = os.getenv("TAVILY_API_KEY")
        if not tavily_api_key:
            raise ValueError("TAVILY_API_KEY not set")

        tavily_client = TavilyClient(api_key=tavily_api_key)
        response = tavily_client.search(query=query, max_results=3)
        for result in response.get("results", []):
            web_results.append({"url": result["url"], "content": result.get("content", "")[:500]})
    except Exception as e:
        print(f"Web search error: {e}")
        return []

    return web_results

In [4]:
import requests


@tool
def fetch_url(url: str) -> dict:
    """Fetch content from a specific URL.

    Args:
        url: The URL to fetch content from

    Returns:
        A dictionary with the URL and its content
    """
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        response = requests.get(url, timeout=10, headers=headers)
        response.raise_for_status()
        return {"url": url, "content": response.text[:5000]}
    except Exception as e:
        error_msg = f"Error fetching {url}: {e}"
        print(error_msg)
        return {"url": url, "error": error_msg}

Bind the tools to the LLM, so it can decide when to call each one:

In [5]:
from langchain_openai import ChatOpenAI

tools = [search_wikipedia, search_web, fetch_url]

# Falls back to a dummy key so the client can be constructed even when OPENAI_API_KEY
# is not set; agent_node() below stops it from ever being called for real.
llm = ChatOpenAI(model="gpt-4o", temperature=0, api_key=os.getenv("OPENAI_API_KEY", "sk-not-set"))
llm_with_tools = llm.bind_tools(tools)

**Agent node** (the supervisor): decides which tool to call next, or ends the conversation.

In [6]:
SYSTEM_PROMPT = """You are a research assistant that helps users gather information on topics.

Your workflow:
1. Use search_wikipedia to get encyclopedia-style information
2. Use search_web to get current web information
3. Use fetch_url if you need to get more details from a specific URL
4. After gathering enough information, synthesize it into a comprehensive report

Be thorough but concise. Always cite your sources."""


def agent_node(state: AgentState) -> dict:
    """The agent node that decides which tools to use."""
    messages = state.messages

    # Add the system prompt on the first turn
    if len(messages) == 1:
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    if not os.getenv("OPENAI_API_KEY"):
        return {"messages": [AIMessage(content="[Skipped LLM call: set OPENAI_API_KEY to run this agent for real]")]}

    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

In [7]:
from langgraph.prebuilt import ToolNode

# Create the tool node that executes tools
tool_node = ToolNode(tools)


def should_continue(state: AgentState) -> Literal["tools", "end"]:
    """Determine if we should continue to tools or end."""
    last_message = state.messages[-1]

    # If the LLM makes tool calls, route to the tools node
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    # Otherwise, end the workflow
    return "end"

In [8]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")

workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "end": END},
)

# Tools always route back to the agent
workflow.add_edge("tools", "agent")

# Compile the graph with checkpointing
checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

In [9]:
config = {"configurable": {"thread_id": "no-mcp-agent-1"}}
result = app.invoke(
    {"messages": [HumanMessage(content="What is the Model Context Protocol?")]},
    config=config,
)
print(result["messages"][-1].content)

[Skipped LLM call: set OPENAI_API_KEY to run this agent for real]
